In [50]:
import json
import os
from tqdm import tqdm
import re

test_path = '/data4/polyakov/instruction_tuning/ToolBench/data/answer/test_toollama_2_ilseyar_test_toolbench_key'
ground_truth_test_path = '/data4/polyakov/instruction_tuning/ToolBench/data/toolbench_multistep_dataset_g_query_format_100_correct_names.json'

In [51]:
ground_truth = json.load(open(ground_truth_test_path, 'r'))

responses = [0] * len(ground_truth)
for filename in os.listdir(test_path):
    query_number = int(filename.split('_')[0])
    data = json.load(open(test_path + '/' + filename, 'r'))
    responses[query_number] = data

for gt, resp in zip(ground_truth, responses):
    assert gt['query'] == resp['answer_generation']['query']

In [52]:
def list_api_called(item, keep_finish=False):
    trys = item['trys']
    assert len(trys) == 1
    chain = trys[0]['chain']
    api_list = []
    for message in chain:
        if message['node_type'] == 'Action':
            curr_api = message['description']
            if curr_api == 'Finish':
                if keep_finish:
                    api_list.append([curr_api, ''])
                else:
                    break
            else:
                api_list.append(curr_api.split('_for_')[::-1])
        if message['node_type'] == 'Action Input':
            api_list[-1].append(message['description'])
            api_list[-1].append(message['observation'])
    return api_list

In [53]:
all_api_calls = [list_api_called(item) for item in responses]
all_api_calls[17]

[['deezer',
  'track',
  '{\n  "is_id": "1115229"\n}',
  '{"error": "", "response": "{\'id\': 1115229, \'readable\': True, \'title\': \\"Big Girls Don\'t Cry (Personal)\\", \'title_short\': \\"Big Girls Don\'t Cry (Personal)\\", \'title_version\': \'\', \'isrc\': \'USUM70609115\', \'link\': \'https://www.deezer.com/track/1115229\', \'share\': \'https://www.deezer.com/track/1115229?utm_source=deezer&utm_content=track-1115229&utm_term=0_1698151989&utm_medium=web\', \'duration\': 268, \'track_position\': 1, \'disk_number\': 1, \'rank\': 157638, \'release_date\': \'2007-07-09\', \'explicit_lyrics\': False, \'explicit_content_lyrics\': 0, \'explicit_content_cover\': 2, \'preview\': \'https://cdns-preview-8.dzcdn.net/stream/c-8e6232d1fa49cf04927f9ee2801405cf-7.mp3\', \'bpm\': 113, \'gain\': -8.9, \'available_countries\': [\'AO\', \'AR\', \'AU\', \'BD\', \'BE\', \'BF\', \'BG\', \'BI\', \'BJ\', \'CD\', \'CF\', \'CG\', \'CH\', \'CI\', \'CM\', \'CZ\', \'DJ\', \'DK\', \'EE\', \'ER\', \'ES\', \'ET

In [54]:
num_tools_called = 0
num_repetitions = 0
for item in all_api_calls:
    for i, api_call in enumerate(item):
        if i > 0:
            if api_call[0] == item[i-1][0] and api_call[1] == item[i-1][1] and api_call[2] == item[i-1][2]:
                print(api_call)
                print(item[i-1])
                print('#' * 100)
                num_repetitions += 1
        num_tools_called += 1


['body_mass_index_bmi_calculator', 'weight_category', '{\n  "bmi": 0.0025390625\n}', '{"error": "", "response": "{\'bmi\': \'0.0025390625\', \'weightCategory\': \'Under Weight\'}"}']
['body_mass_index_bmi_calculator', 'weight_category', '{\n  "bmi": 0.0025390625\n}', '{"error": "", "response": "{\'bmi\': \'0.0025390625\', \'weightCategory\': \'Under Weight\'}"}']
####################################################################################################
['body_mass_index_bmi_calculator', 'weight_category', '{\n  "bmi": 0.004571428571428572\n}', '{"error": "", "response": "{\'bmi\': \'0.004571428571428572\', \'weightCategory\': \'Under Weight\'}"}']
['body_mass_index_bmi_calculator', 'weight_category', '{\n  "bmi": 0.004571428571428572\n}', '{"error": "", "response": "{\'bmi\': \'0.004571428571428572\', \'weightCategory\': \'Under Weight\'}"}']
####################################################################################################
['body_mass_index_bmi_calculator',

In [55]:
num_repetitions / num_tools_called

0.3630952380952381